In [1]:
import pandas as pd
import numpy as np
import matplotlib as mtl
import seaborn as sb

In [3]:
df = pd.read_csv("dataset/Attacks_AppOnly_02-03-2026.csv",sep=';', low_memory=False)

In [4]:
df.shape

(3070159, 52)

In [5]:
df.columns

Index(['id', 'headache', 'migraine', 'attack_time', 'intensity', 'duration',
       'symptoms_both_side_headache', 'symptoms_one_side_headache',
       'symptoms_pulsating_headache', 'symptoms_dull_headache',
       'symptoms_increased_during_physical_activity', 'symptoms_nausea',
       'symptoms_vomiting', 'symptoms_light_sensitivity',
       'symptoms_noise_sensitivity', 'symptoms_smell_sensitivity',
       'symptoms_vertigo', 'symptoms_concentration_disorder',
       'symptoms_tiredness', 'symptoms_aura', 'symptoms_aura_vision',
       'symptoms_aura_emotion', 'symptoms_aura_speech',
       'symptoms_aura_paralysis', 'effects_unfit_for_work',
       'effects_unfit_for_household_recreation', 'effects_doctor',
       'effects_emergency_room', 'effects_hospital', 'comment_menstruation',
       'patient_info_id', 'source_version_id', 'api_version_id', 'create_date',
       'modified_date', 'nMed', 'Triptane_Geb', 'Triptane_Eff',
       'EinfacheSM_Geb', 'EinfacheSM_Eff', 'Uebelkeit_Geb

In [35]:
from ydata_profiling import ProfileReport
report = ProfileReport(df, title="Post-Merge Profile", explorative=True)
report.to_file("profile_report_app_attack.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 52/52 [00:44<00:00,  1.17it/s][A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.groupby(["patient_info_id"]).size().sort_values(ascending=False)

patient_info_id
45861150    1974
58668734    1937
65018782    1636
37360157    1474
98600514    1350
            ... 
32092068       1
32095145       1
32098888       1
31947019       1
78308651       1
Length: 58811, dtype: int64

In [21]:
relevant_columns=["id","patient_info_id","headache","migraine","attack_time","duration","symptoms_both_side_headache", "symptoms_one_side_headache",
       "symptoms_pulsating_headache", "symptoms_dull_headache","weekday"]
req_df=df[relevant_columns].copy()

In [22]:
req_df.head()

,id,patient_info_id,headache,migraine,attack_time,duration,symptoms_both_side_headache,symptoms_one_side_headache,symptoms_pulsating_headache,symptoms_dull_headache,weekday
0,117,72741515,True,False,2020-06-08,480,False,False,True,False,Montag
1,118,72741515,True,True,2020-06-09,360,False,False,True,False,Dienstag
2,127,44203359,True,True,2020-06-19,300,False,True,True,False,Freitag
3,128,44203359,True,True,2020-06-20,300,False,True,True,False,Samstag
4,129,44203359,True,True,2020-06-21,480,False,True,True,False,Sonntag


In [23]:
#checking the missing value in the df
req_df.isnull().sum()

id                             0
patient_info_id                0
headache                       0
migraine                       0
attack_time                    0
duration                       0
symptoms_both_side_headache    0
symptoms_one_side_headache     0
symptoms_pulsating_headache    0
symptoms_dull_headache         0
weekday                        0
dtype: int64

In [24]:
req_df.isna().any()

id                             False
patient_info_id                False
headache                       False
migraine                       False
attack_time                    False
duration                       False
symptoms_both_side_headache    False
symptoms_one_side_headache     False
symptoms_pulsating_headache    False
symptoms_dull_headache         False
weekday                        False
dtype: bool

In [25]:
req_df.dtypes

id                              int64
patient_info_id                 int64
headache                         bool
migraine                         bool
attack_time                    object
duration                        int64
symptoms_both_side_headache      bool
symptoms_one_side_headache       bool
symptoms_pulsating_headache      bool
symptoms_dull_headache           bool
weekday                        object
dtype: object

In [31]:
#parsed = pd.to_datetime(df["date"], errors="coerce")
#parsed.isna()

req_df["attack_time"]=pd.to_datetime(req_df["attack_time"],format="%Y-%m-%d")

In [33]:
req_df["weekday"].unique()

array(['Montag', 'Dienstag', 'Freitag', 'Samstag', 'Sonntag', 'Mittwoch',
       'Donnerstag'], dtype=object)

In [34]:
order = ["Montag","Dienstag","Mittwoch","Donnerstag","Freitag","Samstag","Sonntag"]
req_df["weekday"] = pd.Categorical(req_df["weekday"], categories=order, ordered=True)
req_df.dtypes

id                                      int64
patient_info_id                         int64
headache                                 bool
migraine                                 bool
attack_time                    datetime64[ns]
duration                                int64
symptoms_both_side_headache              bool
symptoms_one_side_headache               bool
symptoms_pulsating_headache              bool
symptoms_dull_headache                   bool
weekday                              category
parsed                         datetime64[ns]
dtype: object

In [36]:
#check the numeric columns for irregularity

# Convert to numeric, invalid entries become NaN
req_df["duration_check"] = pd.to_numeric(req_df["duration"], errors='coerce')

# Rows where conversion failed
invalid_rows = req_df[req_df["duration_check"].isna()]
print("Rows with invalid duration values:")
print(invalid_rows)

Rows with invalid duration values:
Empty DataFrame
Columns: [id, patient_info_id, headache, migraine, attack_time, duration, symptoms_both_side_headache, symptoms_one_side_headache, symptoms_pulsating_headache, symptoms_dull_headache, weekday, parsed, duration_check]
Index: []


In [37]:
invalid_weekday = req_df[~req_df["weekday"].apply(lambda x: isinstance(x, str))]
print("Rows with non-string weekday:")
print(invalid_weekday)

Rows with non-string weekday:
Empty DataFrame
Columns: [id, patient_info_id, headache, migraine, attack_time, duration, symptoms_both_side_headache, symptoms_one_side_headache, symptoms_pulsating_headache, symptoms_dull_headache, weekday, parsed, duration_check]
Index: []
